In [1]:
import json
import os
import time

from typing import TypedDict

from dotenv import load_dotenv

from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_mistralai import ChatMistralAI
from langchain_community.tools.tavily_search import TavilySearchResults

from langgraph.graph import StateGraph

C:\Users\praveen.vanaparthy\AppData\Local\Temp\ipykernel_25236\1273551314.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [2]:
load_dotenv()

MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

print("Mistral key loaded:", bool(MISTRAL_API_KEY))
print("Tavily key loaded:", bool(TAVILY_API_KEY))

Mistral key loaded: True
Tavily key loaded: True


In [3]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.5,
    check_every_n_seconds=0.1,
    max_bucket_size=1,
)

print("Rate limiter created successfully")

Rate limiter created successfully


In [4]:
llm = ChatMistralAI(
    model="ministral-14b-2512",
    temperature=0.7,
    max_retries=2,
    timeout=120,
    rate_limiter=rate_limiter,
)

print("Mistral LLM initialized successfully")

Mistral LLM initialized successfully


In [5]:
tavily_search = TavilySearchResults(
    api_key=TAVILY_API_KEY
)

print("Tavily search initialized successfully")

Tavily search initialized successfully


C:\Users\praveen.vanaparthy\AppData\Local\Temp\ipykernel_25236\360673198.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(


In [6]:
class HealthBotState(TypedDict):
    health_topic: str
    search_results: str
    summary: str
    quiz_set: list
    answers: dict
    feedback: str
    score: int
    continue_session: bool

print("HealthBotState created successfully")

HealthBotState created successfully


In [7]:
def call_llm(prompt: str) -> str:
    for attempt in range(3):
        try:
            response = llm.invoke(prompt)
            return response.content

        except Exception as e:
            if attempt == 2:
                raise

            wait_time = 2 ** attempt
            print(f"Retrying after {wait_time} seconds...")
            time.sleep(wait_time)

    return ""

In [8]:
def build_summary(health_topic: str, search_results: str) -> str:
    prompt = f"""
You are a health education assistant.

Health topic:
{health_topic}

Medical search results:
{search_results}

Using ONLY the information from the medical search results, create a
patient-friendly summary in 3–4 paragraphs.

Do not diagnose the user.
Do not invent information.
Clearly mention when professional medical advice may be needed.
"""

    return call_llm(prompt)

In [27]:
def parse_quiz_json(text: str) -> dict:
    text = text.strip()

    if text.startswith("```"):
        text = text.replace("```json", "").replace("```", "").strip()

    try:
        quiz = json.loads(text)

        if not isinstance(quiz, dict):
            raise ValueError("Quiz response is not a JSON object")

        return quiz

    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")

        if start != -1 and end != -1:
            quiz = json.loads(text[start:end + 1])

            if not isinstance(quiz, dict):
                raise ValueError("Quiz response is not a JSON object")

            return quiz

        raise ValueError("Could not parse quiz JSON")

In [10]:
def build_quiz_set(health_topic: str, summary: str) -> list:
    prompt = f"""
Create exactly 3 multiple-choice questions about this health topic:

{health_topic}

Use ONLY the following summary:

{summary}

Return ONLY valid JSON in this exact format:

[
  {{
    "question": "Question text",
    "options": {{
      "A": "Option A",
      "B": "Option B",
      "C": "Option C",
      "D": "Option D"
    }},
    "correct_option": "A"
  }}
]

Rules:
- Exactly 3 questions
- Exactly 4 options per question: A, B, C, D
- correct_option must be A, B, C, or D
- No markdown
- No explanation outside the JSON
"""

    quiz = parse_quiz_json(call_llm(prompt))

    if len(quiz) != 3:
        raise ValueError("Quiz must contain exactly 3 questions")

    for question in quiz:
        if set(question["options"].keys()) != {"A", "B", "C", "D"}:
            raise ValueError("Each question must have A, B, C, and D options")

    return quiz

In [11]:
def grade_mcq(
    question: str,
    options: dict,
    correct_option: str,
    user_answer: str,
    summary: str,
) -> str:

    prompt = f"""
You are grading a health education quiz.

Summary:
{summary}

Question:
{question}

Options:
{options}

Correct answer:
{correct_option}

User answer:
{user_answer}

Based ONLY on the summary, provide brief feedback.

If correct, say why it is correct.
If incorrect, give the correct answer and briefly explain why.

Do not provide diagnosis or personalized medical advice.
"""

    return call_llm(prompt)

In [12]:
def get_health_topic(state: HealthBotState) -> HealthBotState:
    topic = input("Enter a health topic: ").strip()

    if not topic:
        raise ValueError("Health topic cannot be empty")

    state["health_topic"] = topic
    return state

In [13]:
def search_medical_info(state: HealthBotState) -> HealthBotState:
    topic = state["health_topic"]

    query = f"{topic} medical information reputable sources"

    results = tavily_search.invoke({
        "query": query
    })

    state["search_results"] = json.dumps(results, indent=2)

    return state

In [14]:
def summarize_results(state: HealthBotState) -> HealthBotState:
    summary = build_summary(
        state["health_topic"],
        state["search_results"]
    )

    state["summary"] = summary

    return state

In [28]:
def generate_quiz(state: HealthBotState):
    quiz_prompt = f"""
You are creating a patient education comprehension check.

Use ONLY the following summary as your source of information.

SUMMARY:
{state["summary"]}

Create EXACTLY ONE multiple-choice question.

Requirements:
- The question must be answerable using ONLY the summary.
- Provide exactly four options: A, B, C, D.
- Provide the correct answer.
- Do not use any information outside the summary.

Return JSON only in this exact format:

{{
    "question": "Your question",
    "options": {{
        "A": "Option A",
        "B": "Option B",
        "C": "Option C",
        "D": "Option D"
    }},
    "answer": "A"
}}
"""

    response = call_llm(quiz_prompt)
    quiz = parse_quiz_json(response)

    state["quiz_set"] = [quiz]

    return state

In [29]:
def present_quiz(state: HealthBotState) -> HealthBotState:
    print("\n" + "=" * 60)
    print(f"Health Topic: {state['health_topic']}")
    print("=" * 60)

    print("\nSummary:\n")
    print(state["summary"])

    print("\n" + "=" * 60)
    print("COMPREHENSION CHECK")
    print("=" * 60)

    ready = input("\nAre you ready for the comprehension check? (yes/no): ").strip().lower()

    if ready not in ["yes", "y"]:
        print("\nPlease review the summary and continue when you are ready.")
        return state

    question = state["quiz_set"][0]

    print(f"\n1. {question['question']}")

    for option, text in question["options"].items():
        print(f"   {option}. {text}")

    answer = input("\nYour answer (A/B/C/D): ").strip().upper()
    state["answers"][1] = answer

    return state


In [24]:
def grade_quiz(state: HealthBotState):
    quiz = state["quiz_set"][0]
    question = quiz["question"]
    correct_answer = quiz["answer"]
    user_answer = state["answers"].get(0, "")

    grading_prompt = f"""
You are grading a patient education quiz.

Use ONLY the SUMMARY below to evaluate the patient's answer.

SUMMARY:
{state["summary"]}

QUESTION:
{question}

CORRECT ANSWER:
{correct_answer}

PATIENT ANSWER:
{user_answer}

Assign one grade:
A = Completely correct
B = Mostly correct
C = Incorrect

Return your response in exactly this format:

Grade: A/B/C
Explanation: <brief explanation based ONLY on the summary>
"""

    response = call_llm(grading_prompt)

    state["feedback"] = response

    print("\n" + "=" * 50)
    print("COMPREHENSION CHECK RESULT")
    print("=" * 50)
    print(response)
    print("=" * 50)

    return state

In [18]:
def ask_continue(state: HealthBotState) -> HealthBotState:
    print("\n" + "=" * 60)
    print(f"Your score: {state['score']} / {len(state['quiz_set'])}")
    print("=" * 60)

    print("\nFeedback:\n")
    print(state["feedback"])

    choice = input("\nDo you want to continue with another topic? (yes/no): ")
    state["continue_session"] = choice.strip().lower() in {"yes", "y"}

    return state

In [19]:
def reset_state(state: HealthBotState) -> HealthBotState:
    return {
        "health_topic": "",
        "search_results": "",
        "summary": "",
        "quiz_set": [],
        "answers": {},
        "feedback": "",
        "score": 0,
        "continue_session": False,
    }

In [30]:
graph_builder = StateGraph(HealthBotState)

graph_builder.add_node("get_topic", get_health_topic)
graph_builder.add_node("search", search_medical_info)
graph_builder.add_node("summarize", summarize_results)
graph_builder.add_node("quiz", generate_quiz)
graph_builder.add_node("present_quiz", present_quiz)
graph_builder.add_node("grade", grade_quiz)
graph_builder.add_node("ask_continue", ask_continue)
graph_builder.add_node("reset", reset_state)

graph_builder.set_entry_point("get_topic")

graph_builder.add_edge("get_topic", "search")
graph_builder.add_edge("search", "summarize")
graph_builder.add_edge("summarize", "quiz")
graph_builder.add_edge("quiz", "present_quiz")
graph_builder.add_edge("present_quiz", "grade")
graph_builder.add_edge("grade", "ask_continue")

graph_builder.add_conditional_edges(
    "ask_continue",
    lambda state: "reset" if state["continue_session"] else "__end__",
)

graph_builder.add_edge("reset", "get_topic")

graph = graph_builder.compile()

print("LangGraph compiled successfully")

LangGraph compiled successfully


In [31]:
initial_state: HealthBotState = {
    "health_topic": "",
    "search_results": "",
    "summary": "",
    "quiz_set": [],
    "answers": {},
    "feedback": "",
    "score": 0,
    "continue_session": False,
}

result = graph.invoke(initial_state)

print("\nHealthBot session completed.")

Retrying langchain_mistralai.chat_models.ChatMistralAI.completion_with_retry.<locals>._completion_with_retry in 4 seconds as it raised ConnectError: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1010).



Health Topic: Vitamin D

Summary:

Since the medical search results you provided did not yield usable information (due to a technical error), I’ll share a **general, evidence-based summary** of Vitamin D based on widely accepted health guidelines. If you’d like, I can also suggest reliable sources where you can find more details.

---

**What Is Vitamin D?**
Vitamin D is a nutrient that plays a crucial role in keeping your bones, teeth, and muscles healthy. Unlike most vitamins, your body can produce it naturally when your skin is exposed to sunlight. It also comes from certain foods (like fatty fish, fortified dairy, and egg yolks) and supplements. Many people don’t get enough vitamin D, especially in winter or in regions with limited sunlight. This can lead to deficiencies, which may affect bone health, immune function, and even mood.

**Why Is Vitamin D Important?**
Vitamin D helps your body absorb calcium, which is essential for strong bones and preventing conditions like osteopor